In [ ]:
from tapas_gmm.dataset.scene import SceneDataset
from pathlib import Path
from tapas_gmm.dataset.bc import BCDataset, BCDataConfig
from torch.utils.data import DataLoader
from tapas_gmm.utils.observation import collate
from tapas_gmm.dataset.demos import Demos

from tapas_gmm.policy.models.tpgmm import (
    AutoTPGMM,
    AutoTPGMMConfig,
    TPGMMConfig,
    FrameSelectionConfig,
    DemoSegmentationConfig,
    InitStrategy,
    FittingStage,
)

2026-06-10 11:18:06.189 | INFO     |  Running on cpu


/home/nils/Documents/Study Project/Code/riepybdlib/riepybdlib/data.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_listdir


In [2]:
data_root = Path("../outputs/bimanual_dataset")

In [3]:
loaded_dataset = SceneDataset(
    data_root=Path(data_root)
)

2026-06-10 11:18:08.610 | INFO     |  Initializing datasete using ../outputs/bimanual_dataset/metadata.json
2026-06-10 11:18:08.612 | INFO     |  Extracted gt object labels []
2026-06-10 11:18:08.612 | INFO     |  Extracted tsdf object labels []


In [4]:
bc_config = BCDataConfig(
    fragment_length=-1,
    cameras = tuple(),
)

bc_dataset = BCDataset(
    scene_dataset=loaded_dataset,
    config=bc_config,
)

2026-06-10 11:18:08.625 | INFO     |  Initializing BCDataset:
2026-06-10 11:18:08.625 | INFO     |    Training on fragments of length -1.
2026-06-10 11:18:08.625 | INFO     |    Loading raw data for encoder.


In [5]:
loader = DataLoader(
    bc_dataset,
    collate_fn=collate,
)

In [6]:
def make_arm_traj(traj, arm):
    arm_traj = traj.clone()

    if arm == "left":
        arm_traj.ee_pose = traj.ee_pose[:, :7]
        arm_traj.gripper_state = traj.gripper_state[:, :1]
        arm_traj.action = traj.action[:, :7]

    elif arm == "right":
        arm_traj.ee_pose = traj.ee_pose[:, 7:]
        arm_traj.gripper_state = traj.gripper_state[:, 1:]
        arm_traj.action = traj.action[:, 7:]

    return arm_traj

In [7]:
left_trajs = []
right_trajs = []

for batch in loader:
    sample = batch[0]

    left_traj = make_arm_traj(sample, "left")
    right_traj = make_arm_traj(sample, "right")

    left_trajs.append(left_traj)
    right_trajs.append(right_traj)

left_demos = Demos(left_trajs)
right_demos = Demos(right_trajs)

2026-06-10 11:18:09.514 | INFO     |  Subsampling to length 104 using strategy mean-length.


In [8]:
obs0 = left_trajs[0][0]

for name, pose in obs0.object_poses.items():
    print(name, pose)

obj002 tensor([ 0.4379, -0.0476,  0.7605,  0.9720,  0.0000,  0.0000,  0.2350])
obj000 tensor([ 0.1956, -0.3304,  0.7605,  0.7735,  0.0000,  0.0000,  0.6337])
obj001 tensor([0.2613, 0.3443, 0.7605, 0.5653, 0.0000, 0.0000, 0.8249])


In [ ]:
tpgmm_config = TPGMMConfig(
    add_time_component=True,
)

segmentation_config = DemoSegmentationConfig(
    no_segmentation=True,
)

auto_config = AutoTPGMMConfig(
    tpgmm=tpgmm_config,
    demos_segmentation=segmentation_config,
)


In [10]:
left_model = AutoTPGMM(auto_config)

left_lik, left_avg_loglik = left_model.fit_trajectories(
    left_demos,
    fix_frames=True,
    fitting_actions=[
        FittingStage.INIT,
        FittingStage.EM_HMM,
    ],
)

print(left_avg_loglik)

2026-06-10 11:18:09.568 | INFO     |  Fitting AutoTPGMM
2026-06-10 11:18:09.569 | INFO     |  Performing fitting actions: [INIT, EM_HMM]
2026-06-10 11:18:09.569 | INFO     |  Segmenting trajectories
2026-06-10 11:18:09.569 | INFO     |  ... created 1 segments
2026-06-10 11:18:09.569 | INFO     |    Fitting candidate frame 1/5
2026-06-10 11:18:09.569 | INFO     |    Creating partial frame view of demos.
2026-06-10 11:18:09.585 | INFO     |    Manifold: TIME x R3 x QUAT
2026-06-10 11:18:09.585 | INFO     |    Changing number of components to 4
2026-06-10 11:18:09.585 | INFO     |    Init strategy not specified. Auto selected InitStrategy.TIME_BASED.
2026-06-10 11:18:09.585 | INFO     |    Model init ...


Time-based init:   0%|          | 0/4 [00:00<?, ?it/s]

2026-06-10 11:18:09.745 | INFO     |    HMM EM ...
2026-06-10 11:18:09.750 | INFO     |    HMM transition matrix not defined, initializing to uniform


HMM EM:   0%|          | 0/50 [00:00<?, ?it/s]

2026-06-10 11:18:09.783 | INFO     |    HMM init priors not defined, initializing to uniform
2026-06-10 11:18:13.624 | INFO     |    HMM EM converged
2026-06-10 11:18:13.653 | INFO     |    Fitting candidate frame 2/5
2026-06-10 11:18:13.653 | INFO     |    Creating partial frame view of demos.


...

Fitting segments:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-10 11:18:20.474 | INFO     |  Assuming zero frame velocity. Should be fixed.
2026-06-10 11:18:20.475 | INFO     |  Manifold: TIME x R3 x QUAT x R3 x QUAT x R3 x QUAT x R3 x QUAT


Time-based init:   0%|          | 0/4 [00:00<?, ?it/s]

HMM EM:   0%|          | 0/50 [00:00<?, ?it/s]

(6894.042754071599,)
